In [77]:
# Initialize Git in this folder
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /home/jovyan/work/group project/.git/


In [78]:
#import relevant libraries 

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

In [79]:
#import data and wrangle data 
players_untidy=pd.read_csv("players.csv")
experience_mapping = {'Veteran': 5, 'Pro': 4, 'Amateur': 1, 'Regular': 3, 'Beginner': 2}
players_untidy['experience_enc'] = players_untidy['experience'].map(experience_mapping)


players=players_untidy[["age","experience_enc","subscribe"]]

players

,age,experience_enc,subscribe
0,9,4,True
1,17,5,True
2,17,5,False
3,21,1,True
4,21,3,True
...,...,...,...
191,17,1,True
192,22,5,False
193,17,1,False
194,17,1,False


In [80]:
#split data, use a random state 
players_train, players_test = train_test_split(players, test_size=0.25, random_state=123)

X_train = players_train[["experience_enc", "age"]]
y_train = players_train["subscribe"]

X_test = players_test[["experience_enc", "age"]]
y_test = players_test["subscribe"]

In [81]:
#change experience to a numerical variable + standardize training data 
players_preprocessor = make_column_transformer(
    (StandardScaler(), ["age","experience_enc"]),
     verbose_feature_names_out=False
)

In [82]:
#find best k using CV 
param_grid = {
    "kneighborsclassifier__n_neighbors": range(2, 15, 1),
}

players_pipe = make_pipeline(players_preprocessor, KNeighborsClassifier())

knn_tune_grid = GridSearchCV(
    players_pipe, param_grid, cv=4,
)
knn_model_grid = knn_tune_grid.fit(X_train, y_train)

accuracies_grid = pd.DataFrame(knn_model_grid.cv_results_)

In [83]:
#plot to visualize best k 
accuracy_versus_k_grid = alt.Chart(accuracies_grid).mark_line(point=True).encode(
    x=alt.X("param_kneighborsclassifier__n_neighbors")
        .title("Neighbors")
        .scale(zero=False),
    y=alt.Y("mean_test_score")
        .title("Average Validation Accuracy")
        .scale(zero=False)
)

#confirm best k
knn_tune_grid.best_params_

{'kneighborsclassifier__n_neighbors': 13}

In [84]:
#run the model on the test data 

players_test["predicted"] = knn_tune_grid.predict(
    players_test[["experience_enc", "age"]]
)

knn_tune_grid.score(
   players_test[["experience_enc", "age"]],
   players_test["subscribe"]
)

precision_score(
    y_true=players_test["subscribe"],
    y_pred=players_test["predicted"],
    pos_label=True
)

recall_score(
    y_true=players_test["subscribe"],
    y_pred=players_test["predicted"],
    pos_label=True
)

pd.crosstab(
    players_test["subscribe"],
    players_test["predicted"]
)

predicted,True
subscribe,
False,15
True,34


In [85]:
#evaluate model
# 1. Create the grid of feature values
exp_grid = np.linspace(
    players_test[["experience_enc"]].min().iloc[0] * 0.95,
    players_test[["experience_enc"]].max().iloc[0] * 1.05,
    50
)

age_grid = np.linspace(
    players_test["age"].min() * 0.95,
    players_test["age"].max() * 1.05,
    50
)

grid = np.array(np.meshgrid(exp_grid, age_grid)).reshape(2, -1).T
grid = pd.DataFrame(grid, columns=["experience_enc", "age"])


# 2. Predict at the grid points
grid_preds = knn_tune_grid.predict(grid)

# 3. Bind predictions to the grid
prediction_table = grid.copy()
prediction_table["predicted"] = grid_preds


# 4. Plot original data
data_plot = alt.Chart(players_test).mark_point(
    opacity=0.6,
    filled=True,
    size=40
).encode(
    x=alt.X(
        "experience_enc",
        scale=alt.Scale(
            nice=False,
            domain=[
                players_test["experience_enc"].min() * 0.95,
                players_test["experience_enc"].max() * 1.05
            ]
        )
    ),
    y=alt.Y(
        "age",
        scale=alt.Scale(
            nice=False,
            domain=[
                players_test["age"].min() * 0.95,
                players_test["age"].max() * 1.05
            ]
        )
    ),
    color=alt.Color("subscribe:N", title="Actual")
)


# 5. Plot prediction background
prediction_plot = alt.Chart(prediction_table).mark_point(
    opacity=0.05,
    filled=True,
    size=300
).encode(
    x="experience_enc",
    y="age",
    color=alt.Color("predicted:N", title="Predicted")
)


# 6. Combine plots
data_plot + prediction_plot

alt.LayerChart(...)